In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from stuff import (
    regrid_from_norkyst,
)

BASEDIR = Path(os.environ["PROJECT_ROOT"])
input_dir = BASEDIR / "data" / "input"
output_dir = BASEDIR / "data" / "output"

In [3]:

PARAMETERS = ["temperature", "salinity", "u_eastward", "v_northward"]

### Regridding
Open the downloaded datasets and regrid them to the coordinates from a bathymetry file.

In [9]:
FILES = sorted(
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.startswith("NorKyst-800m_ZDEPTHS_avg_")
)
FILES

['/Users/jemmima/dev/gcp-chem-sim-private/data/input/NorKyst-800m_ZDEPTHS_avg_2020.nc']

In [10]:
ds_grid = xr.open_dataset(os.path.join(input_dir,"bathymetry_19to67.nc"))

In [11]:
lat_diff = np.diff(ds_grid["lat"].values)[0]
lat_faces = np.zeros(shape=ds_grid["lat"].values.shape[0] + 1)
lat_faces[0] = ds_grid["lat"].values[0] - lat_diff / 2
lat_faces[1:] = ds_grid["lat"].values + lat_diff / 2

In [12]:
lon_diff = np.diff(ds_grid["lon"].values)[0]
lon_faces = np.zeros(shape=ds_grid["lon"].values.shape[0] + 1)
lon_faces[0] = ds_grid["lon"].values[0] - lon_diff / 2
lon_faces[1:] = ds_grid["lon"].values + lon_diff / 2

In [13]:
z_faces = ds_grid.z_faces.values
z_centers = [(z_faces[i] + z_faces[i + 1]) / 2 for i in range(len(z_faces) - 1)]
ds_out_c = xr.Dataset(
    {
        "lat": (["lat"], ds_grid["lat"].values, {"units": "degrees_north"}),
        "lon": (["lon"], ds_grid["lon"].values, {"units": "degrees_east"}),
    }
)
ds_out_u = xr.Dataset(
    {
        "lat": (["lat"], ds_grid["lat"].values, {"units": "degrees_north"}),
        "lon": (["lon"], lon_faces, {"units": "degrees_east"}),
    }
)
ds_out_v = xr.Dataset(
    {
        "lat": (["lat"], lat_faces, {"units": "degrees_north"}),
        "lon": (["lon"], ds_grid["lon"].values, {"units": "degrees_east"}),
    }
)

In [ ]:
# Concatenate all files first, then regrid once (much faster than per-file regridding)
dss_in = []
for file in FILES:
    ds_in = xr.open_dataset(file)
    new_time = pd.date_range(start=ds_in.time[0].values, end=ds_in.time[-1].values, freq="D")
    ds_in = ds_in.reindex(time=new_time, method="ffill")
    dss_in.append(ds_in)
    print(f"File {file} loaded.")

ds_all = xr.concat(dss_in, dim="time")
print(f"Concatenated {len(FILES)} files: {ds_all.sizes}")

regridder_rho, regridder_u, regridder_v, np_time, np_temp, np_salt, np_u, np_v = regrid_from_norkyst(
    None, None, None, ds_all, ds_out_c, ds_out_u, ds_out_v, np.array(z_centers)
)
print("Regridding complete.")


File /Users/jemmima/dev/gcp-chem-sim-private/data/input/NorKyst-800m_ZDEPTHS_avg_2020.nc loaded.


In [ ]:
np_temp = np_temp.astype(np.float32)
np_salt = np_salt.astype(np.float32)
np_u = np_u.astype(np.float32)
np_v = np_v.astype(np.float32)


In [ ]:
Tout_lambda = np.zeros_like(np_temp)
Sout_lambda = np.zeros_like(np_salt)
Uout_lambda = np.zeros_like(np_u)
Vout_lambda = np.zeros_like(np_v)

In [ ]:
dsout = xr.Dataset(
    {
        "T": (["time", "Nz", "Ny", "Nx"], np_temp),
        "T_lambda": (["time", "Nz", "Ny", "Nx"], Tout_lambda),
        "S": (["time", "Nz", "Ny", "Nx"], np_salt),
        "S_lambda": (["time", "Nz", "Ny", "Nx"], Sout_lambda),
        "u": (["time", "Nz", "Ny", "Nx_faces"], np_u),
        "u_lambda": (["time", "Nz", "Ny", "Nx_faces"], Uout_lambda),
        "v": (["time", "Nz", "Ny_faces", "Nx"], np_v),
        "v_lambda": (["time", "Nz", "Ny_faces", "Nx"], Vout_lambda),
    },
    coords={
        "time": np_time,
        "Nz": z_centers,
        "Nz_faces": z_faces,
        "Ny": ds_grid["lat"].values,
        "Ny_faces": lat_faces,
        "Nx": ds_grid["lon"].values,
        "Nx_faces": lon_faces,
    },
)

In [ ]:
dsout

In [ ]:
z_level = -1

In [ ]:
dsout.T.isel(time=0, Nz=z_level).plot()

In [ ]:
dsout.S.isel(time=0, Nz=z_level).plot()

In [ ]:
dsout.u.isel(time=0, Nz=z_level).plot()

In [ ]:
dsout.v.isel(time=0, Nz=z_level).plot()

In [ ]:
encoding = {var: {"zlib": True, "complevel": 5} for var in dsout.data_vars}
dsout.to_netcdf(
    os.path.join(input_dir, f"regridded_19to67.nc"),
    encoding=encoding,
)